In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('2019-Nov.csv')

In [3]:
df.shape
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 67501979 entries, 0 to 67501978
Data columns (total 9 columns):
 #   Column         Dtype  
---  ------         -----  
 0   event_time     str    
 1   event_type     str    
 2   product_id     int64  
 3   category_id    int64  
 4   category_code  str    
 5   brand          str    
 6   price          float64
 7   user_id        int64  
 8   user_session   str    
dtypes: float64(1), int64(3), str(5)
memory usage: 4.5 GB


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-11-01 00:00:00 UTC,view,1003461,2053013555631882655,electronics.smartphone,xiaomi,489.07,520088904,4d3b30da-a5e4-49df-b1a8-ba5943f1dd33
1,2019-11-01 00:00:00 UTC,view,5000088,2053013566100866035,appliances.sewing_machine,janome,293.65,530496790,8e5f4f83-366c-4f70-860e-ca7417414283
2,2019-11-01 00:00:01 UTC,view,17302664,2053013553853497655,NaN,creed,28.31,561587266,755422e7-9040-477b-9bd2-6a6e8fd97387
3,2019-11-01 00:00:01 UTC,view,3601530,2053013563810775923,appliances.kitchen.washer,lg,712.87,518085591,3bfb58cd-7892-48cc-8020-2f17e6de6e7f
4,2019-11-01 00:00:01 UTC,view,1004775,2053013555631882655,electronics.smartphone,xiaomi,183.27,558856683,313628f1-68b8-460d-84f6-cec7a8796ef2


In [4]:
df.isnull().sum().sort_values(ascending=False)

category_code    21898171
brand             9224078
user_session           10
event_time              0
event_type              0
category_id             0
product_id              0
price                   0
user_id                 0
dtype: int64

In [8]:
# 1. Drop dữ liệu quan trọng
df = df.dropna(subset=['price', 'user_id'])

# 2. Xử lý brand
df['brand'] = df['brand'].fillna('unknown')

# 3. KHÔNG dùng category_code
# (giữ nguyên nhưng không phân tích)

In [ ]:
# xử lý giá trị bất thường
# kiểm tra xem giá trị price có phân bố như thế nào

df['price'].describe()

count    6.750198e+07
mean     2.924593e+02
std      3.556745e+02
min      0.000000e+00
25%      6.924000e+01
50%      1.657700e+02
75%      3.603400e+02
max      2.574070e+03
Name: price, dtype: float64

In [ ]:
# có nhiều giá trị price bằng 0, có thể là lỗi dữ liệu hoặc sản phẩm miễn phí
(df['price'] == 0).sum()

np.int64(188088)

In [14]:
# xem xét price bằng 0 có trong các sự kiện nào 
df[df['price'] == 0]['event_type'].value_counts()


event_type
view    183680
cart      4408
Name: count, dtype: int64

In [19]:
# nếu price bằng 0 chủ yếu là view hoặc cart, có thể là lỗi dữ liệu, nên loại bỏ
df = df[df['price'] > 0]

In [20]:
# kiểm tra lại sau khi loại bỏ
df[df['price'] == 0]['event_type'].value_counts()

Series([], Name: count, dtype: int64)

In [21]:
# có nhiều giá trị price bằng 0, có thể là lỗi dữ liệu hoặc sản phẩm miễn phí
(df['price'] == 0).sum()

np.int64(0)

In [26]:
# check lại lỗi ở đâu
print("Min price:", df['price'].min())
print("Count price = 0:", (df['price'] == 0).sum())
print("Shape:", df.shape)

Min price: 0.77
Count price = 0: 0
Shape: (67313891, 9)


In [23]:
# kiểm tra 
df['event_type'].value_counts()

event_type
view        63372430
cart         3024522
purchase      916939
Name: count, dtype: int64

In [27]:
df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-11-01 00:00:00 UTC,view,1003461,2053013555631882655,electronics.smartphone,xiaomi,489.07,520088904,4d3b30da-a5e4-49df-b1a8-ba5943f1dd33
1,2019-11-01 00:00:00 UTC,view,5000088,2053013566100866035,appliances.sewing_machine,janome,293.65,530496790,8e5f4f83-366c-4f70-860e-ca7417414283
2,2019-11-01 00:00:01 UTC,view,17302664,2053013553853497655,NaN,creed,28.31,561587266,755422e7-9040-477b-9bd2-6a6e8fd97387
3,2019-11-01 00:00:01 UTC,view,3601530,2053013563810775923,appliances.kitchen.washer,lg,712.87,518085591,3bfb58cd-7892-48cc-8020-2f17e6de6e7f
4,2019-11-01 00:00:01 UTC,view,1004775,2053013555631882655,electronics.smartphone,xiaomi,183.27,558856683,313628f1-68b8-460d-84f6-cec7a8796ef2


In [24]:
# lọc lại dữ liệu chỉ giữ lại 3 loại sự kiện: view, cart, purchase
df = df[df['event_type'].isin(['view','cart','purchase'])]

In [25]:
# kiểm tra rồi chuyển sang phân tích tỷ lệ phần trăm    
df['event_type'].value_counts(normalize=True)

event_type
view        0.941447
cart        0.044932
purchase    0.013622
Name: proportion, dtype: float64

In [28]:
# tính tỷ lệ phần trăm thêm vào giỏ hàng và tỷ lệ phần trăm mua hàng
view = df[df['event_type']=='view'].shape[0]
cart = df[df['event_type']=='cart'].shape[0]
purchase = df[df['event_type']=='purchase'].shape[0]

cart_rate = cart / view
purchase_rate = purchase / cart
print(f"Cart rate: {cart_rate:.2%}")
print(f"Purchase rate: {purchase_rate:.2%}")

Cart rate: 4.77%
Purchase rate: 30.32%


In [ ]:

df['price'].describe()

count    6.750198e+07
mean     2.924593e+02
std      3.556745e+02
min      0.000000e+00
25%      6.924000e+01
50%      1.657700e+02
75%      3.603400e+02
max      2.574070e+03
Name: price, dtype: float64

In [11]:
df['price'].quantile([0.9, 0.95, 0.99])

0.90     756.75
0.95     992.16
0.99    1661.12
Name: price, dtype: float64

In [12]:
df = df[df['price'] < df['price'].quantile(0.99)]

In [12]:
missing_ratio = df.isnull().mean().sort_values(ascending=False)


df.isnull().sum().head()

event_time              0
event_type              0
product_id              0
category_id             0
category_code    21898171
dtype: int64

In [7]:
dtypes = {
    "event_type": "category",
    "product_id": "int32",
    "category_id": "float32",
    "category_code": "str",
    "brand": "str",
    "price": "float32",
    "user_id": "int32",
    "user_session": "str"
}

In [8]:
chunks = []

for chunk in pd.read_csv(
    file_path,
    chunksize=chunk_size,
    dtype=dtypes,
    parse_dates=["event_time"]
):
    
    # 🔥 1. Xóa dòng thiếu user_id hoặc product_id
    chunk = chunk.dropna(subset=["user_id", "product_id"])
    
    # 🔥 2. Xóa duplicate
    chunk = chunk.drop_duplicates()
    
    # 🔥 3. Xử lý missing brand
    chunk["brand"] = chunk["brand"].fillna("unknown")
    
    # 🔥 4. Xóa giá <= 0
    chunk = chunk[chunk["price"] > 0]
    
    # 🔥 5. Tạo cột doanh thu (chỉ khi purchase)
    chunk["revenue"] = np.where(
        chunk["event_type"] == "purchase",
        chunk["price"],
        0
    )
    
    # 🔥 6. Tách thời gian (feature engineering nhẹ)
    chunk["hour"] = chunk["event_time"].dt.hour
    chunk["day"] = chunk["event_time"].dt.day
    
    chunks.append(chunk)

In [11]:
df = pd.concat(chunks, ignore_index=True)

In [13]:
df.info()
df.describe()
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 67213478 entries, 0 to 67213477
Data columns (total 12 columns):
 #   Column         Dtype              
---  ------         -----              
 0   event_time     datetime64[us, UTC]
 1   event_type     str                
 2   product_id     int32              
 3   category_id    float32            
 4   category_code  str                
 5   brand          str                
 6   price          float32            
 7   user_id        int32              
 8   user_session   str                
 9   revenue        float32            
 10  hour           int32              
 11  day            int32              
dtypes: datetime64[us, UTC](1), float32(3), int32(4), str(4)
memory usage: 4.3 GB


event_time              0
event_type              0
product_id              0
category_id             0
category_code    21775316
brand                   0
price                   0
user_id                 0
user_session           10
revenue                 0
hour                    0
day                     0
dtype: int64